# Issue #15 Phase 2: Locked 2024 Preflight Reproduction Gate

This notebook contains only the safety checks and locked 2024 reproduction gate required before the 2025 out-of-time holdout may be accessed.

**Safety boundary:** the 2025 CSV is not opened, parsed, previewed, summarized, or evaluated here. This notebook does not contain a cell that loads it. No 2025 metrics are calculated. The saved pipeline is used only through its fitted inference paths; neither `fit` nor `fit_transform` is called.

Only aggregate counts and metrics are displayed. Complaint narratives, normalized-text hashes, row-level predictions, and decision scores remain local and are neither displayed nor saved.

In [1]:
from pathlib import Path
import ast
import hashlib
import inspect
import platform
import subprocess
import sys
import warnings

import joblib
import nbformat
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.utils.validation import check_is_fitted


def find_project_root(start_path: Path) -> Path:
    current = start_path.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() and (candidate / "reports" / "2025_validation_protocol.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root and committed validation protocol.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.routing_rules import AUTO_ROUTE, HUMAN_REVIEW, route_from_scores


NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "07_2025_out_of_time_validation.ipynb"
PROTOCOL_PATH = PROJECT_ROOT / "reports" / "2025_validation_protocol.md"
MODEL_PATH = PROJECT_ROOT / "models" / "best_tfidf_classifier.joblib"
REFERENCE_2024_PATH = PROJECT_ROOT / "data" / "processed" / "cfpb_complaints_2024_cleaned.csv"
HOLDOUT_2025_PATH = PROJECT_ROOT / "data" / "raw" / "cfpb_complaints_2025_raw.csv"
ROUTING_SOURCE_PATH = PROJECT_ROOT / "src" / "routing_rules.py"

ALLOWED_BRANCHES = ("issue-15-2025-holdout-evaluation", "main")
BASELINE_COMMIT = "fddca32f466737901bbf95534d08030b32e9599e"
EXPECTED_HOLDOUT_2025_SIZE = 73_806_040
EXPECTED_HOLDOUT_2025_SHA256 = "b59d7842e786f00d6be26b7980a42f67474acb9040db293ddd3641204d25eb3a"
EXPECTED_VERSIONS = {
    "Python": "3.11.15",
    "Scikit-learn": "1.9.0",
    "Pandas": "3.0.3",
    "NumPy": "2.4.6",
}
EXPECTED_FILES = {
    "Locked model": {
        "path": MODEL_PATH,
        "size": 3_392_109,
        "sha256": "4514e7e49e305e408e2eaaf296d8607b33e9320547685339eff263e4dda0c94a",
    },
    "Locked 2024 cleaned reference": {
        "path": REFERENCE_2024_PATH,
        "size": 54_908_639,
        "sha256": "b115eb0c4a20a881a6a45bfb74cb7d715a726537372baa7d68f09d657cdfd919",
    },
}
EXPECTED_CLASSES = (
    "Checking or savings account",
    "Credit card",
    "Credit reporting or other personal consumer reports",
    "Debt collection",
    "Money transfer, virtual currency, or money service",
    "Mortgage",
    "Student loan",
    "Vehicle loan or lease",
)
EXPECTED_MODELING_ROWS = 33_042
EXPECTED_DEVELOPMENT_ROWS = 26_433
EXPECTED_FINAL_TEST_ROWS = 6_609
OUTER_SPLITS = 5
FINAL_TEST_FOLD = 0
RANDOM_STATE = 42
MIN_TOP_SCORE = 0.08
MIN_SCORE_MARGIN = 0.73

EXPECTED_CLASSIFICATION = {
    "Accuracy": 0.8712,
    "Macro precision": 0.7734,
    "Macro recall": 0.7621,
    "Macro F1": 0.7671,
    "Weighted precision": 0.8721,
    "Weighted recall": 0.8712,
    "Weighted F1": 0.8715,
}
EXPECTED_ROUTING = {
    "Auto-routed rows": 5_092,
    "Human-review rows": 1_517,
    "Auto-routing coverage": 0.7705,
    "Human-review rate": 0.2295,
    "Auto-routed accuracy": 0.9503,
    "Auto-routed misroute rate": 0.0497,
}

check_records = []


def record_check(name: str, passed: bool, observed: object, expected: object) -> None:
    check_records.append(
        {
            "check": name,
            "status": "PASS" if bool(passed) else "FAIL",
            "observed": str(observed),
            "expected": str(expected),
        }
    )


def stop_if_failed(section: str) -> None:
    failures = [row for row in check_records if row["status"] == "FAIL"]
    if failures:
        display(pd.DataFrame(failures))
        raise RuntimeError(
            f"{section} failed. Stop before accessing the 2025 holdout; do not change the model or protocol."
        )


print(f"Project root located: {PROJECT_ROOT}")
print("2025 holdout access: DISABLED for this notebook")
print("Output policy: aggregate checks and metrics only")

Project root located: E:\MGA\ITEC6740\Final-Project\financial-complaint-auto-routing-nlp
2025 holdout access: DISABLED for this notebook
Output policy: aggregate checks and metrics only


## 1. Safety, branch, protocol, and environment checks

The notebook first verifies its execution boundary and exact artifact-compatible environment. It also confirms that the current protocol file is the same Git blob recorded on `origin/main`.

In [2]:
def run_git(*args: str, check: bool = True) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        ["git", *args],
        cwd=PROJECT_ROOT,
        check=check,
        capture_output=True,
        text=True,
    )


current_branch = run_git("branch", "--show-current").stdout.strip()
record_check("Current branch is allowed", current_branch in ALLOWED_BRANCHES, current_branch, ALLOWED_BRANCHES)

protocol_relative = PROTOCOL_PATH.relative_to(PROJECT_ROOT).as_posix()
remote_protocol_blob = run_git("rev-parse", f"origin/main:{protocol_relative}").stdout.strip()
local_protocol_blob = run_git("hash-object", protocol_relative).stdout.strip()
record_check(
    "Protocol committed and present on origin/main",
    local_protocol_blob == remote_protocol_blob,
    local_protocol_blob,
    remote_protocol_blob,
)

baseline_is_ancestor = run_git("merge-base", "--is-ancestor", BASELINE_COMMIT, "HEAD", check=False).returncode == 0
record_check(
    "Locked baseline commit is an ancestor of HEAD",
    baseline_is_ancestor,
    baseline_is_ancestor,
    True,
)

observed_versions = {
    "Python": platform.python_version(),
    "Scikit-learn": sklearn.__version__,
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
}
for component, expected_version in EXPECTED_VERSIONS.items():
    observed_version = observed_versions[component]
    record_check(
        f"{component} version",
        observed_version == expected_version,
        observed_version,
        expected_version,
    )

record_check("2025 holdout access enabled", False is False, False, False)
stop_if_failed("Safety and environment checks")

display(
    pd.DataFrame(
        [{"component": key, "observed": value, "expected": EXPECTED_VERSIONS[key]} for key, value in observed_versions.items()]
    )
)
print("Safety, Git, and environment checks passed.")

,component,observed,expected
0,Python,3.11.15,3.11.15
1,Scikit-learn,1.9.0,1.9.0
2,Pandas,3.0.3,3.0.3
3,NumPy,2.4.6,2.4.6


Safety, Git, and environment checks passed.


In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


integrity_rows = []
for label, specification in EXPECTED_FILES.items():
    path = specification["path"]
    exists = path.is_file()
    record_check(f"{label} path exists", exists, exists, True)
    if not exists:
        continue

    observed_size = path.stat().st_size
    observed_sha256 = sha256_file(path)
    size_matches = observed_size == specification["size"]
    hash_matches = observed_sha256 == specification["sha256"]
    record_check(f"{label} file size", size_matches, observed_size, specification["size"])
    record_check(f"{label} SHA-256", hash_matches, observed_sha256, specification["sha256"])
    integrity_rows.append(
        {
            "file": path.relative_to(PROJECT_ROOT).as_posix(),
            "size_bytes": observed_size,
            "size_status": "PASS" if size_matches else "FAIL",
            "sha256": observed_sha256,
            "sha256_status": "PASS" if hash_matches else "FAIL",
        }
    )

stop_if_failed("Locked file-integrity checks")
display(pd.DataFrame(integrity_rows))
print("Only the locked model and 2024 cleaned reference were fingerprinted in this notebook.")

,file,size_bytes,size_status,sha256,sha256_status
0,models/best_tfidf_classifier.joblib,3392109,PASS,4514e7e49e305e408e2eaaf296d8607b33e93205476853...,PASS
1,data/processed/cfpb_complaints_2024_cleaned.csv,54908639,PASS,b115eb0c4a20a881a6a45bfb74cb7d715a726537372baa...,PASS


Only the locked model and 2024 cleaned reference were fingerprinted in this notebook.


In [4]:
with warnings.catch_warnings(record=True) as load_warnings:
    warnings.simplefilter("always")
    pipeline = joblib.load(MODEL_PATH)

record_check("Model compatibility warnings", len(load_warnings) == 0, len(load_warnings), 0)
record_check("Loaded object is sklearn Pipeline", isinstance(pipeline, Pipeline), type(pipeline).__name__, "Pipeline")

try:
    check_is_fitted(pipeline)
    fitted_pipeline = True
except Exception:
    fitted_pipeline = False
record_check("Pipeline is fitted", fitted_pipeline, fitted_pipeline, True)

expected_step_names = ["tfidf", "classifier"]
observed_step_names = list(pipeline.named_steps) if isinstance(pipeline, Pipeline) else []
record_check("Pipeline step names", observed_step_names == expected_step_names, observed_step_names, expected_step_names)

tfidf = pipeline.named_steps["tfidf"]
classifier = pipeline.named_steps["classifier"]
record_check("TF-IDF step type", isinstance(tfidf, TfidfVectorizer), type(tfidf).__name__, "TfidfVectorizer")
record_check("Classifier step type", isinstance(classifier, LinearSVC), type(classifier).__name__, "LinearSVC")

expected_tfidf_params = {
    "input": "content",
    "encoding": "utf-8",
    "decode_error": "strict",
    "strip_accents": None,
    "lowercase": True,
    "preprocessor": None,
    "tokenizer": None,
    "analyzer": "word",
    "token_pattern": r"(?u)\b\w\w+\b",
    "ngram_range": (1, 2),
    "stop_words": None,
    "max_df": 1.0,
    "min_df": 2,
    "max_features": 50_000,
    "vocabulary": None,
    "binary": False,
    "dtype": np.float64,
    "norm": "l2",
    "use_idf": True,
    "smooth_idf": True,
    "sublinear_tf": False,
}
expected_svm_params = {
    "penalty": "l2",
    "loss": "squared_hinge",
    "dual": "auto",
    "tol": 0.0001,
    "C": 1.0,
    "multi_class": "ovr",
    "fit_intercept": True,
    "intercept_scaling": 1,
    "class_weight": "balanced",
    "verbose": 0,
    "random_state": 42,
    "max_iter": 10_000,
}


def parameter_values_match(observed: object, expected: object) -> bool:
    if expected is np.float64:
        return np.dtype(observed) == np.dtype(np.float64)
    return observed == expected


observed_tfidf_params = tfidf.get_params(deep=False)
tfidf_mismatches = {
    name: (observed_tfidf_params[name], expected)
    for name, expected in expected_tfidf_params.items()
    if not parameter_values_match(observed_tfidf_params[name], expected)
}
observed_svm_params = classifier.get_params(deep=False)
svm_mismatches = {
    name: (observed_svm_params[name], expected)
    for name, expected in expected_svm_params.items()
    if not parameter_values_match(observed_svm_params[name], expected)
}
record_check("Locked TF-IDF parameters", not tfidf_mismatches, tfidf_mismatches or "all matched", "all matched")
record_check("Locked Linear SVM parameters", not svm_mismatches, svm_mismatches or "all matched", "all matched")

fitted_vocabulary_length = len(tfidf.vocabulary_)
fitted_idf_length = len(tfidf.idf_)
record_check("Fitted TF-IDF vocabulary length", fitted_vocabulary_length == 50_000, fitted_vocabulary_length, 50_000)
record_check("Fitted TF-IDF IDF length", fitted_idf_length == 50_000, fitted_idf_length, 50_000)

pipeline_class_order = tuple(pipeline.classes_)
record_check("Fitted classes_ order", pipeline_class_order == EXPECTED_CLASSES, pipeline_class_order, EXPECTED_CLASSES)

routing_source = Path(inspect.getsourcefile(route_from_scores)).resolve()
record_check("Existing route_from_scores implementation", routing_source == ROUTING_SOURCE_PATH.resolve(), routing_source, ROUTING_SOURCE_PATH.resolve())
routing_signature = inspect.signature(route_from_scores)
required_routing_parameters = {"class_labels", "decision_scores", "min_top_score", "min_score_margin"}
record_check(
    "Routing function parameters",
    required_routing_parameters.issubset(routing_signature.parameters),
    tuple(routing_signature.parameters),
    tuple(sorted(required_routing_parameters)),
)
record_check("Locked top-score threshold", MIN_TOP_SCORE == 0.08, MIN_TOP_SCORE, 0.08)
record_check("Locked score-margin threshold", MIN_SCORE_MARGIN == 0.73, MIN_SCORE_MARGIN, 0.73)

stop_if_failed("Locked pipeline checks")
print("Locked fitted TF-IDF + Linear SVM pipeline checks passed.")
print(f"Pipeline steps: {observed_step_names}")
print(f"Fitted TF-IDF features: {fitted_vocabulary_length:,}")
print(f"Fitted classes: {len(pipeline_class_order)}")
print("Model loaded without compatibility warnings.")

Locked fitted TF-IDF + Linear SVM pipeline checks passed.
Pipeline steps: ['tfidf', 'classifier']
Fitted TF-IDF features: 50,000
Fitted classes: 8
Model loaded without compatibility warnings.


## 2. Reconstruct the locked 2024 reference partitions

The reconstruction uses only the fingerprinted 2024 cleaned reference. Hashes are temporary in-memory group identifiers and are never displayed or saved.

In [5]:
required_columns = ["clean_complaint_text", "product"]
source_df = pd.read_csv(REFERENCE_2024_PATH, usecols=required_columns)

missing_columns = sorted(set(required_columns).difference(source_df.columns))
missing_value_count = int(source_df[required_columns].isna().sum().sum()) if not missing_columns else -1
record_check("2024 required columns", not missing_columns, missing_columns or "present", "present")
record_check("2024 required-field missing values", missing_value_count == 0, missing_value_count, 0)
stop_if_failed("2024 cleaned-reference schema checks")


def normalize_cleaned_text(value: object) -> str:
    return " ".join(str(value).strip().split())


def stable_text_hash(value: object) -> str:
    normalized = normalize_cleaned_text(value)
    if not normalized:
        raise ValueError("Normalized complaint text must not be empty.")
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


working_df = source_df[required_columns].copy()
working_df["normalized_text_hash"] = working_df["clean_complaint_text"].map(stable_text_hash)

label_counts_by_hash = working_df.groupby("normalized_text_hash", sort=False)["product"].nunique()
conflicting_hashes = set(label_counts_by_hash[label_counts_by_hash > 1].index)

locked_scope_df = working_df[working_df["product"].isin(EXPECTED_CLASSES)].copy()
locked_scope_df = locked_scope_df[
    ~locked_scope_df["normalized_text_hash"].isin(conflicting_hashes)
].copy()
remediated_df = locked_scope_df.drop_duplicates(
    subset=["normalized_text_hash", "product"],
    keep="first",
).reset_index(drop=True)

outer_splitter = StratifiedGroupKFold(
    n_splits=OUTER_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)
outer_folds = list(
    outer_splitter.split(
        remediated_df["clean_complaint_text"],
        remediated_df["product"],
        groups=remediated_df["normalized_text_hash"],
    )
)
development_indices, final_test_indices = outer_folds[FINAL_TEST_FOLD]
development_df = remediated_df.iloc[development_indices].reset_index(drop=True)
final_test_df = remediated_df.iloc[final_test_indices].reset_index(drop=True)

development_groups = set(development_df["normalized_text_hash"])
final_test_groups = set(final_test_df["normalized_text_hash"])
overlap_count = len(development_groups.intersection(final_test_groups))
development_classes = set(development_df["product"].unique())
final_test_classes = set(final_test_df["product"].unique())
expected_class_set = set(EXPECTED_CLASSES)

record_check("Corrected 2024 modeling rows", len(remediated_df) == EXPECTED_MODELING_ROWS, len(remediated_df), EXPECTED_MODELING_ROWS)
record_check("2024 modeling hashes are unique", remediated_df["normalized_text_hash"].is_unique, remediated_df["normalized_text_hash"].is_unique, True)
record_check("2024 development rows", len(development_df) == EXPECTED_DEVELOPMENT_ROWS, len(development_df), EXPECTED_DEVELOPMENT_ROWS)
record_check("2024 final internal-test rows", len(final_test_df) == EXPECTED_FINAL_TEST_ROWS, len(final_test_df), EXPECTED_FINAL_TEST_ROWS)
record_check("All eight classes in development", development_classes == expected_class_set, len(development_classes), len(expected_class_set))
record_check("All eight classes in final internal test", final_test_classes == expected_class_set, len(final_test_classes), len(expected_class_set))
record_check("Development/final-test normalized-text overlap", overlap_count == 0, overlap_count, 0)
stop_if_failed("Locked 2024 reconstruction")

display(
    pd.DataFrame(
        [
            {"aggregate": "Corrected modeling rows", "observed": len(remediated_df), "expected": EXPECTED_MODELING_ROWS},
            {"aggregate": "Development rows", "observed": len(development_df), "expected": EXPECTED_DEVELOPMENT_ROWS},
            {"aggregate": "Final internal-test rows", "observed": len(final_test_df), "expected": EXPECTED_FINAL_TEST_ROWS},
            {"aggregate": "Classes in each partition", "observed": len(development_classes), "expected": len(expected_class_set)},
            {"aggregate": "Normalized-text overlap", "observed": overlap_count, "expected": 0},
        ]
    )
)
print("Locked 2024 reference reconstruction passed.")

,aggregate,observed,expected
0,Corrected modeling rows,33042,33042
1,Development rows,26433,26433
2,Final internal-test rows,6609,6609
3,Classes in each partition,8,8
4,Normalized-text overlap,0,0


Locked 2024 reference reconstruction passed.


## 3. Reproduce locked 2024 classification results

The already fitted pipeline evaluates only the reconstructed final internal-test rows. No model training or parameter selection occurs.

In [6]:
y_final = final_test_df["product"].to_numpy()
final_predictions = pipeline.predict(final_test_df["clean_complaint_text"])

macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_final,
    final_predictions,
    labels=list(pipeline_class_order),
    average="macro",
    zero_division=0,
)
weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
    y_final,
    final_predictions,
    labels=list(pipeline_class_order),
    average="weighted",
    zero_division=0,
)
observed_classification = {
    "Accuracy": float(accuracy_score(y_final, final_predictions)),
    "Macro precision": float(macro_precision),
    "Macro recall": float(macro_recall),
    "Macro F1": float(macro_f1),
    "Weighted precision": float(weighted_precision),
    "Weighted recall": float(weighted_recall),
    "Weighted F1": float(weighted_f1),
}

classification_rows = []
for metric, expected_value in EXPECTED_CLASSIFICATION.items():
    observed_value = observed_classification[metric]
    passed = round(observed_value, 4) == expected_value
    record_check(f"2024 classification: {metric}", passed, f"{observed_value:.4f}", f"{expected_value:.4f}")
    classification_rows.append(
        {
            "metric": metric,
            "observed": f"{observed_value:.4f}",
            "expected": f"{expected_value:.4f}",
            "status": "PASS" if passed else "FAIL",
        }
    )

stop_if_failed("Locked 2024 classification reproduction")
display(pd.DataFrame(classification_rows))
print("All locked 2024 classification metrics reproduced to four decimal places.")

,metric,observed,expected,status
0,Accuracy,0.8712,0.8712,PASS
1,Macro precision,0.7734,0.7734,PASS
2,Macro recall,0.7621,0.7621,PASS
3,Macro F1,0.7671,0.7671,PASS
4,Weighted precision,0.8721,0.8721,PASS
5,Weighted recall,0.8712,0.8712,PASS
6,Weighted F1,0.8715,0.8715,PASS


All locked 2024 classification metrics reproduced to four decimal places.


## 4. Reproduce locked 2024 routing results

Routing uses the fitted pipeline's decision function and class order, the existing `route_from_scores` implementation, and the locked inclusive thresholds. Decision scores are model signals, not probabilities, and are never displayed or saved.

In [7]:
final_score_matrix = pipeline.decision_function(final_test_df["clean_complaint_text"])
score_shape_ok = final_score_matrix.shape == (EXPECTED_FINAL_TEST_ROWS, len(pipeline_class_order))
finite_scores = bool(np.isfinite(final_score_matrix).all())
record_check("Final-test decision-score matrix shape", score_shape_ok, final_score_matrix.shape, (EXPECTED_FINAL_TEST_ROWS, len(pipeline_class_order)))
record_check("Final-test decision scores are finite", finite_scores, finite_scores, True)
stop_if_failed("Locked 2024 decision-score checks")

final_decisions = [
    route_from_scores(
        pipeline_class_order,
        score_row,
        min_top_score=MIN_TOP_SCORE,
        min_score_margin=MIN_SCORE_MARGIN,
    )
    for score_row in final_score_matrix
]
final_auto_mask = np.asarray(
    [decision["routing_decision"] == AUTO_ROUTE for decision in final_decisions],
    dtype=bool,
)
final_review_mask = np.asarray(
    [decision["routing_decision"] == HUMAN_REVIEW for decision in final_decisions],
    dtype=bool,
)
routed_predictions = np.asarray([decision["predicted_label"] for decision in final_decisions], dtype=object)
prediction_alignment = bool(np.array_equal(routed_predictions, final_predictions))
routing_partition_complete = bool(np.all(final_auto_mask ^ final_review_mask))
record_check("Routing predictions align with pipeline predictions", prediction_alignment, prediction_alignment, True)
record_check("Every row has exactly one routing outcome", routing_partition_complete, routing_partition_complete, True)

final_is_correct = final_predictions == y_final
auto_routed_rows = int(final_auto_mask.sum())
human_review_rows = int(final_review_mask.sum())
auto_routing_coverage = auto_routed_rows / EXPECTED_FINAL_TEST_ROWS
human_review_rate = human_review_rows / EXPECTED_FINAL_TEST_ROWS
auto_routed_accuracy = float(final_is_correct[final_auto_mask].mean())
auto_routed_misroute_rate = 1.0 - auto_routed_accuracy

observed_routing = {
    "Auto-routed rows": auto_routed_rows,
    "Human-review rows": human_review_rows,
    "Auto-routing coverage": auto_routing_coverage,
    "Human-review rate": human_review_rate,
    "Auto-routed accuracy": auto_routed_accuracy,
    "Auto-routed misroute rate": auto_routed_misroute_rate,
}

routing_rows = []
for metric, expected_value in EXPECTED_ROUTING.items():
    observed_value = observed_routing[metric]
    if isinstance(expected_value, int):
        passed = observed_value == expected_value
        observed_display = f"{observed_value:,}"
        expected_display = f"{expected_value:,}"
    else:
        passed = round(observed_value, 4) == expected_value
        observed_display = f"{observed_value:.4f}"
        expected_display = f"{expected_value:.4f}"
    record_check(f"2024 routing: {metric}", passed, observed_display, expected_display)
    routing_rows.append(
        {
            "metric": metric,
            "observed": observed_display,
            "expected": expected_display,
            "status": "PASS" if passed else "FAIL",
        }
    )

stop_if_failed("Locked 2024 routing reproduction")
display(pd.DataFrame(routing_rows))
print("All locked 2024 routing counts and metrics reproduced.")

# Remove row-level labels, predictions, temporary hashes, routing decisions, and scores from memory.
del final_score_matrix, final_decisions, routed_predictions, final_auto_mask, final_review_mask, final_is_correct
del final_predictions, y_final, source_df, working_df, locked_scope_df, remediated_df
del development_df, final_test_df, development_groups, final_test_groups, conflicting_hashes
print("Temporary row-level and hash-bearing objects cleared from memory.")

,metric,observed,expected,status
0,Auto-routed rows,"5,092","5,092",PASS
1,Human-review rows,"1,517","1,517",PASS
2,Auto-routing coverage,0.7705,0.7705,PASS
3,Human-review rate,0.2295,0.2295,PASS
4,Auto-routed accuracy,0.9503,0.9503,PASS
5,Auto-routed misroute rate,0.0497,0.0497,PASS


All locked 2024 routing counts and metrics reproduced.
Temporary row-level and hash-bearing objects cleared from memory.


## 5. Final 2024 preflight summary

The final static safety audit verifies that the notebook contains no training call, contains exactly one CSV read targeting the locked 2024 reference variable, and contains no code reference to the locked 2025 filename. The complete PASS/FAIL table below contains only aggregate and configuration evidence.

In [8]:
with NOTEBOOK_PATH.open("r", encoding="utf-8") as notebook_handle:
    notebook_document = nbformat.read(notebook_handle, as_version=4)

code_cells = [cell.source for cell in notebook_document.cells if cell.cell_type == "code"]
forbidden_training_calls = []
csv_read_arguments = []
approved_holdout_hash_calls = []
holdout_parser_calls = []
parser_call_names = {"read_csv", "read_table", "read_fwf", "read_json", "read_parquet"}
for cell_index, source in enumerate(code_cells):
    syntax_tree = ast.parse(source)
    for node in ast.walk(syntax_tree):
        if not isinstance(node, ast.Call):
            continue
        if isinstance(node.func, ast.Attribute):
            call_name = node.func.attr
        elif isinstance(node.func, ast.Name):
            call_name = node.func.id
        else:
            call_name = ""
        if call_name in {"fit", "fit_transform"}:
            forbidden_training_calls.append((cell_index, call_name))
        if call_name == "read_csv" and node.args:
            csv_read_arguments.append(ast.unparse(node.args[0]))
        if call_name == "sha256_file" and node.args and ast.unparse(node.args[0]) == "HOLDOUT_2025_PATH":
            approved_holdout_hash_calls.append(cell_index)
        if call_name in parser_call_names and node.args and "HOLDOUT_2025_PATH" in ast.unparse(node.args[0]):
            holdout_parser_calls.append((cell_index, call_name))

record_check(
    "No fit or fit_transform call in notebook",
    not forbidden_training_calls,
    forbidden_training_calls or "none",
    "none",
)
record_check(
    "Only locked 2024 CSV read is present",
    csv_read_arguments == ["REFERENCE_2024_PATH"],
    csv_read_arguments,
    ["REFERENCE_2024_PATH"],
)

record_check(
    "2025 access is limited to one approved binary SHA-256 call",
    len(approved_holdout_hash_calls) == 1 and not holdout_parser_calls,
    f"binary_hash_calls={len(approved_holdout_hash_calls)}, parser_calls={len(holdout_parser_calls)}",
    "binary_hash_calls=1, parser_calls=0",
)

save_call_names = {"to_csv", "to_parquet", "to_pickle", "to_json", "save", "savez", "dump"}
row_level_save_calls = []
for cell_index, source in enumerate(code_cells):
    syntax_tree = ast.parse(source)
    for node in ast.walk(syntax_tree):
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Attribute) and node.func.attr in save_call_names:
            row_level_save_calls.append((cell_index, node.func.attr))
record_check(
    "No row-level data save call in notebook",
    not row_level_save_calls,
    row_level_save_calls or "none",
    "none",
)

stop_if_failed("Final 2024 preflight safety audit")
preflight_table = pd.DataFrame(check_records)
display(preflight_table)

all_checks_passed = bool(preflight_table["status"].eq("PASS").all())
if not all_checks_passed:
    raise RuntimeError("2024 preflight gate FAILED. The 2025 holdout must remain unopened.")

print(f"Required checks passed: {len(preflight_table)}/{len(preflight_table)}")
print("2024 PREFLIGHT GATE: PASS")
print("The 2025 holdout remains unparsed and unevaluated; only the approved binary integrity check is authorized next.")

,check,status,observed,expected
0,Current branch is allowed,PASS,issue-15-2025-holdout-evaluation,"('issue-15-2025-holdout-evaluation', 'main')"
1,Protocol committed and present on origin/main,PASS,3f7e9b293227e68884ecc2e6e240c01cdaec8d00,3f7e9b293227e68884ecc2e6e240c01cdaec8d00
2,Locked baseline commit is an ancestor of HEAD,PASS,True,True
3,Python version,PASS,3.11.15,3.11.15
4,Scikit-learn version,PASS,1.9.0,1.9.0
5,Pandas version,PASS,3.0.3,3.0.3
6,NumPy version,PASS,2.4.6,2.4.6
7,2025 holdout access enabled,PASS,False,False
8,Locked model path exists,PASS,True,True
9,Locked model file size,PASS,3392109,3392109


Required checks passed: 59/59
2024 PREFLIGHT GATE: PASS
The 2025 holdout remains unparsed and unevaluated; only the approved binary integrity check is authorized next.


## 2025 Holdout File-Integrity Gate

This gate performs only the pre-open integrity verification committed in the validation protocol. The file is read solely as uninterpreted binary bytes to calculate SHA-256. No CSV parsing, column or row inspection, narrative or label inspection, preview, summary, prediction, or 2025 metric calculation occurs.

In [9]:
integrity_check_start = len(check_records)
expected_holdout_relative = Path("data") / "raw" / "cfpb_complaints_2025_raw.csv"
observed_holdout_relative = HOLDOUT_2025_PATH.relative_to(PROJECT_ROOT)
holdout_exists = HOLDOUT_2025_PATH.is_file()
holdout_path_matches = observed_holdout_relative == expected_holdout_relative

record_check(
    "2025 holdout path exists and matches protocol",
    holdout_exists and holdout_path_matches,
    observed_holdout_relative.as_posix(),
    expected_holdout_relative.as_posix(),
)

if holdout_exists:
    observed_holdout_size = HOLDOUT_2025_PATH.stat().st_size
    observed_holdout_sha256 = sha256_file(HOLDOUT_2025_PATH)
else:
    observed_holdout_size = None
    observed_holdout_sha256 = None

holdout_size_matches = observed_holdout_size == EXPECTED_HOLDOUT_2025_SIZE
holdout_sha256_matches = observed_holdout_sha256 == EXPECTED_HOLDOUT_2025_SHA256
record_check(
    "2025 holdout file size",
    holdout_size_matches,
    observed_holdout_size if observed_holdout_size is not None else "not computed",
    EXPECTED_HOLDOUT_2025_SIZE,
)
record_check(
    "2025 holdout SHA-256",
    holdout_sha256_matches,
    "matches locked protocol" if holdout_sha256_matches else "mismatch or not computed",
    "matches locked protocol",
)

stop_if_failed("2025 holdout file-integrity gate")
holdout_integrity_table = pd.DataFrame(check_records[integrity_check_start:])
display(holdout_integrity_table)

entry_check_summary = pd.DataFrame(check_records)
display(entry_check_summary)
all_entry_checks_passed = bool(entry_check_summary["status"].eq("PASS").all())
if not all_entry_checks_passed:
    raise RuntimeError("Phase 2 entry checks FAILED. Do not parse or evaluate the 2025 holdout.")

print("2025 holdout path check: PASS")
print("2025 holdout file-size check: PASS")
print("2025 holdout SHA-256 check: PASS")
print("Existing 2024 preflight checks passed: 59/59")
print(f"Final entry checks passed: {len(entry_check_summary)}/{len(entry_check_summary)}")
print("2025 HOLDOUT FILE-INTEGRITY GATE: PASS")
print("The 2025 file was read only as raw binary bytes; its CSV contents remain unparsed and unevaluated.")

,check,status,observed,expected
0,2025 holdout path exists and matches protocol,PASS,data/raw/cfpb_complaints_2025_raw.csv,data/raw/cfpb_complaints_2025_raw.csv
1,2025 holdout file size,PASS,73806040,73806040
2,2025 holdout SHA-256,PASS,matches locked protocol,matches locked protocol


,check,status,observed,expected
0,Current branch is allowed,PASS,issue-15-2025-holdout-evaluation,"('issue-15-2025-holdout-evaluation', 'main')"
1,Protocol committed and present on origin/main,PASS,3f7e9b293227e68884ecc2e6e240c01cdaec8d00,3f7e9b293227e68884ecc2e6e240c01cdaec8d00
2,Locked baseline commit is an ancestor of HEAD,PASS,True,True
3,Python version,PASS,3.11.15,3.11.15
4,Scikit-learn version,PASS,1.9.0,1.9.0
...,...,...,...,...
57,2025 access is limited to one approved binary ...,PASS,"binary_hash_calls=1, parser_calls=0","binary_hash_calls=1, parser_calls=0"
58,No row-level data save call in notebook,PASS,none,none
59,2025 holdout path exists and matches protocol,PASS,data/raw/cfpb_complaints_2025_raw.csv,data/raw/cfpb_complaints_2025_raw.csv
60,2025 holdout file size,PASS,73806040,73806040


2025 holdout path check: PASS
2025 holdout file-size check: PASS
2025 holdout SHA-256 check: PASS
Existing 2024 preflight checks passed: 59/59
Final entry checks passed: 62/62
2025 HOLDOUT FILE-INTEGRITY GATE: PASS
The 2025 file was read only as raw binary bytes; its CSV contents remain unparsed and unevaluated.
